# Laboratório 15: O Cirurgião Implacável (RFE - Recursive Feature Elimination)
**Disciplina:** Extração e Preparação de Dados (IBM8915)  
**Tópico:** Feature Selection (Seleção de Atributos) Wrapper

No Laboratório 14 nós utilizamos o `SelectFromModel`, um método *Filtro-Intrínseco* que cortou nossas features pela média estatística usando uma árvore genérica. 
Agora subiremos o nível da nossa faxina final usando o método da família *Wrapper* conhecido como **RFE**.\

### 0. A Mesma Base Poluída
Vamos gerar o mesmo Dataset Monstruoso do laboratório anterior (10.000 linhas, 100 Colunas, muitas delas sendo ruídos) e refazer o split de treino/teste.\

In [2]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=10000, n_features=100, n_informative=10, 
                           n_redundant=40, n_repeated=10, n_classes=2, random_state=42)

colunas = [f'Feature_{i}' for i in range(100)]
X_dados = pd.DataFrame(X, columns=colunas)
y_dados = pd.Series(y, name='Target')

# Split Seguro (Para evitar DATA LEAKAGE na hora de selecionar features)
X_treino, X_teste, y_treino, y_teste = train_test_split(X_dados, y_dados, test_size=0.20, random_state=42)

print(f"Matriz de Treino Bruta (A ser operada): {X_treino.shape}")

Matriz de Treino Bruta (A ser operada): (8000, 100)


### 1. A Abordagem Gulosa do RFE
O `RFE` (Eliminação Recursiva de Atributos) treina o modelo com todas as 100 features. 
Em seguida, ele acha a pior de todas... a menos importante. E a mata.
O Dataset fica com 99 features. Ele treina o modelo inteiro **DE NOVO**. Acha a pior, e a mata. 
Ele realiza essa recursão letal incontáveis vezes até o momento em que restam exatamente o número absoluto de features que você mandou ele deixar vivos (`n_features_to_select`).

**Trade-off:** É dezenas de vezes mais demorado, dezenas de vezes mais preciso.\

In [3]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# 1. Novamente, precisamos de um juiz (Inteligência) para julgar cada passada do loop
modelo_juiz = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

# 2. Instanciando o RFE 
# Diretriz agressiva: "Sobra apenas as 10 FEATURES SUPREMAS da tabela!"
rfe = RFE(estimator=modelo_juiz, n_features_to_select=10, step=2)  # step=2 elimina de duas em duas por vez para ser mais rápido

print("O Cirurgião começou a operar... Isso pode demorar vários segundos dependendo da sua máquina.")
X_treino_rfe = rfe.fit_transform(X_treino, y_treino)

print("Operação de abstração cirúrgica finalizada!")
print(f"Matriz de Treino Após o RFE: {X_treino_rfe.shape}")

O Cirurgião começou a operar... Isso pode demorar vários segundos dependendo da sua máquina.
Operação de abstração cirúrgica finalizada!
Matriz de Treino Após o RFE: (8000, 10)


### 2. O Ranking das Features Vencidas
Além de usar o poderoso `.get_support()` para descobrir o nome da elite 10 que você selecionou, o `RFE` possui o parâmetro `.ranking_`, que te rankeia a ordem exata de letalidade das features. 
As 10 que sobreviveram ficam empatadas na patente militar de "Rank 1". Todo o restante fica nas patentes renegadas (2º lugar, 3º lugar, 14º lugar...).\

In [4]:
# Buscando quem sobreviveu e criando DataFrame
nomes_supremos = X_treino.columns[rfe.get_support()]

print("A Elite das 10 Features Vencedoras do RFE:")
print(nomes_supremos.to_list())

# Demonstrando o Ranking final
# Podemos unir a lista completa de features (as 100 iniciais) numa tabela contra as suas patentes (ranking_. Faz sentido?)
ranking_df = pd.DataFrame({
    'Feature_Original': X_treino.columns,
    'Rank (Prioridade)': rfe.ranking_
}).sort_values(by='Rank (Prioridade)')

print("\nRanking Descritivo das Primeiras e Últimas Features no peso da precisão do Algoritmo:")
ranking_df


A Elite das 10 Features Vencedoras do RFE:
['Feature_4', 'Feature_5', 'Feature_17', 'Feature_30', 'Feature_38', 'Feature_44', 'Feature_55', 'Feature_59', 'Feature_60', 'Feature_78']

Ranking Descritivo das Primeiras e Últimas Features no peso da precisão do Algoritmo:


,Feature_Original,Rank (Prioridade)
59,Feature_59,1
38,Feature_38,1
78,Feature_78,1
4,Feature_4,1
5,Feature_5,1
...,...,...
84,Feature_84,44
16,Feature_16,45
46,Feature_46,45
54,Feature_54,46


### 🚀 O Fim da Preparação! O Início da Engenharia (Aula 13)

Eis as vitórias de vocês até aqui nesse curso:

✅ Análisaram Nulos e Ausentes

✅ Imputaram dados faltantes com Simples Estatística e Machine Learning KNN (Aula Anterior)

✅ Traduziram Letras e Sentenças para Bits Matriciais Matemáticos (One Hot Encoding/Ordinal Encoding)

✅ Expansão de Features em Datas e Ceps (Feature Engineering Temporal Exaustiva)

✅ Padronizaram as grandezas para não matar a distância geométrica (Escala/StandardScaler)

✅ Produziram dados Sintéticos para compensar a Armadilha Minoritária Fraudulenta (Balanceamento SMOTE)

✅ Faxina Dimensional! Mataram colunas ruidosas para otimizar processamento, e extraíram as "dez mais valiosas" (RFE e Feature Selection)

O que mais falta a uma estrutura de dados bem tratada nas mãos de um cientista? 
**Mais nada!**

Porém, todos esses códigos foram redigidos de um modo "sujo", passo-a-passo no caderninho Jupyter. E se o sistema da IBM receber um arquivo de Banco de Dados diferente todos os dias da semana às 4 da manhã? Vamos abrir os 15 `labs.ipynb` um a um e apertar 'Shift+Enter' todos os dias da nossa vida? 

É aí que a gente abandona o laboratório solitário do Jupyter Notebook e entra na **Engenharia Contínua de Software**.
Vejo vocês na próxima aula para criarmos **Pipelines Orientados a Objetos do Scikit-Learn**.\

---
## 🚀 Desafio Prático: A Cirurgia Fina de Risco de Crédito

No desafio anterior do SelectFromModel, a árvore escolheu livremente quantas variáveis ela quis cortar da base de Fraude de Cartão de Crédito.
Agora, o banco para o qual você trabalha te deu uma diretriz arquitetônica rigorosa: **"O nosso sistema em produção só aguenta processar 5 features em tempo real durante a transação do cartão"**.

**Sua Missão:**
1. Carregue o seu `X_treino` e `y_treino` do dataset do Kaggle (Credit Card Fraud).
2. Use a força bruta inteligente do `RFE` (Recursive Feature Elimination).
3. Configure o parâmetro obrigatório `n_features_to_select=5`.
4. Deixe o algoritmo treinar, eliminar, treinar e eliminar até sobrarem apenas a Elite das 5 Features Vencedoras.
5. Imprima o `.ranking_` para documentar quem ficou em 1º, 2º, e último lugar.\

In [ ]:
# IMPLEMENTE SEU CÓDIGO AQUI

# 1. Carregue os dados do Kaggle (Credit Card Fraud)
df_kag = pd.read_csv('creditcard.csv')

X_kag = df_kag.drop(columns=['Class'])
y_kag = df_kag['Class']

X_kag_treino, X_kag_teste, y_kag_treino, y_kag_teste = train_test_split(
    X_kag, y_kag, test_size=0.20, random_state=42
)

# 2. RFE com n_features_to_select=5
rfe_kaggle = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    n_features_to_select=5,
    step=5
)

print("Iniciando RFE no dataset do Kaggle... (pode demorar alguns minutos)")
rfe_kaggle.fit(X_kag_treino, y_kag_treino)

# 3. Imprima a elite e o ranking!
elite_5 = X_kag_treino.columns[rfe_kaggle.get_support()]
print(f"\nElite das 5 Features Vencedoras:")
print(elite_5.to_list())

ranking_kag = pd.DataFrame({
    'Feature': X_kag_treino.columns,
    'Rank': rfe_kaggle.ranking_
}).sort_values('Rank')

print("\nRanking completo (Top 10):")
display(ranking_kag.head(10))

---
## 🧠 Questão Discursiva

**Pergunta:** O RFE é um método *Wrapper* da categoria "Guloso" (Greedy). Ele treina o modelo preditivo iterativamente dezenas (ou centenas) de vezes, sempre matando a pior variável do grupo, uma a uma.

Em um ambiente moderno de Big Data (ex: um cluster Spark processando 5 bilhões de registros com 2.000 colunas), qual o perigo sistêmico de tentar aplicar o `RFE` diretamente na base de dados bruta inteira? Qual estratégia **híbrida** de Feature Selection você propõe para mitigar esse gargalo?\

**SUA RESPOSTA AQUI:**
*(Dê um duplo-clique para editar esta célula e escreva sua resposta).*\